# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. This dataset records adoption predictors of indigenous and modern knowledge in rangeland management across Northern Kenya via ordered logistic regression results, structured according to the [Croissant schema](https://mlcommons.org/croissant/).

### Dataset Source
The dataset schema is accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

---

In [ ]:
# Make sure MLcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We load the dataset metadata and record sets using `mlcroissant`. All dataset entities are referenced by their `@id` as defined in the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata summary
md = dataset.metadata
print(f"Dataset name: {md.name}")
print(f"Identifier: {md.identifier}")
print(f"Version: {md.version}")
print(f"Published: {md.datePublished}")
print(f"Description: {md.description[:200]}...")


## 2. Data Overview

Let's review the available record sets (`RecordSet`), including their `@id`, fields, types, and associated distributions.

*Note*: Each field, column, or record set is referenced via its `@id` as per the schema.

In [ ]:
# List record sets, their fields and file sources (by @id)
from pprint import pprint

print("Record sets and fields available:\n---------------------------")

# dataset.metadata.record_sets is the main entry point for schema-level record sets

rs_list = dataset.metadata.record_sets
if not rs_list:
    print("No record sets found defined in Croissant metadata. Trying to load records directly...")
else:
    for rs in rs_list:
        print(f"\nRecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                print(f"    Field @id: {field['@id']}  -- {field.get('name', field['@id'])}")
        if 'fileObject' in rs:
            print(f"  fileObject(s): {[fobj['@id'] for fobj in rs['fileObject']]}")

# Alternatively, try to use mlcroissant's helpers to inspect available RecordSets
print("\n[INFO] Inspecting dataset for record set IDs using mlcroissant:")
rs_ids = dataset.record_set_ids()
pprint(rs_ids)

# For demo, show the first RecordSet's fields (if available)
if rs_ids:
    ex_rs_id = rs_ids[0]
    print(f"\nExample RecordSet: {ex_rs_id}")
    print("Fields for this record set:")
    field_ids = dataset.field_ids(record_set=ex_rs_id)
    for fid in field_ids:
        print(f"  Field @id: {fid}")


## 3. Data Extraction

Let's load data from one or more record sets using their `@id`, and examine their columns (field `@id`s).

In [ ]:
# Get available record set ids
record_set_ids = dataset.record_set_ids()

# For demonstration, fetch their DataFrames
dataframes = {}

for rs_id in record_set_ids:
    print(f"\n--- Loading from RecordSet '@id': {rs_id} ---")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Fields present: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load data for {rs_id}: {e}")

# Pick the first record set with data for continued EDA
primary_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        primary_record_set_id = rid
        break

print(f"\nUsing primary record set for EDA: {primary_record_set_id}\nFields:")
if primary_record_set_id:
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)

We'll perform common processing steps, such as:
- Filtering records based on numeric thresholds
- Normalizing a numeric field
- Grouping by a categorical field

Edit the variables below to use field `@id`s of your choice from the previous data cell.

In [ ]:
# Example: EDA on the main DataFrame

df = dataframes[primary_record_set_id].copy()

# AUTO-DETECT numeric fields for demo (using dtypes)
potential_numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields found: {potential_numeric_fields}")

# Choose a numeric field (@id)
if potential_numeric_fields:
    numeric_field_id = potential_numeric_fields[0]  # Select first numeric field
else:
    raise ValueError("No numeric fields found for EDA.")

threshold = df[numeric_field_id].quantile(0.8)  # Example: filter top 20% of this metric
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (showing top rows):")
print(filtered_df.head(2))

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Select a grouping (categorical) field if any present
cat_fields = df.select_dtypes(include=['object']).columns.tolist()
group_field_id = None
for col in cat_fields:
    if col != numeric_field_id:
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field, and its mean value across groups, if a categorical field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the numeric field's distribution
sns.set(style='whitegrid')
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field is found, plot group means
if group_field_id:
    plt.figure(figsize=(7, 4))
    sns.barplot(y=grouped_df[group_field_id], x=grouped_df[numeric_field_id], color='salmon', orient='h')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We have loaded and previewed the FAIR² dataset's Croissant schema and record sets, accessed all entities via their `@id`s, inspected field details, and extracted tabular data for exploration.
- Exploratory analysis illustrated filtering and normalization for one numeric field, and summarized grouped statistics for a categorical feature if available.
- Visualizations enabled inspection of distributions and group-wise means, providing insight into the structure and trends in the dataset.

**Next steps**: You can extend this notebook for deeper analysis, variable selection (referenced strictly via `@id`), visualizations, and modeling as required for your application.

---